# Imports

In [ ]:
from pprint import pp

from settings import settings

# Modelle

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url=settings.endpoint,
    api_key=settings.api_key
)

response = client.responses.create(
    model=settings.model,
    input="Welcher Tag ist heute?",
)

print(response.output_text)

## Web search

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url=settings.endpoint,
    api_key=settings.api_key
)

response = client.responses.create(
    model=settings.model,
    instructions="Du bist ein frecher AI Assistant.",
    input="Welcher Tag ist heute?",
    max_output_tokens=1000,
    tools=[{"type": "web_search"}]
)

print(response.output_text)

## File_search

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url=settings.endpoint,
    api_key=settings.api_key
)

# Create vector store and upload a file
vector_store = client.vector_stores.create(name="policy-docs")
client.vector_stores.files.upload_and_poll(
    vector_store_id=vector_store.id,
    file=open("expenses_policy.pdf", "rb")
)

# Get response using the file_search tool
response = client.responses.create(
    model=settings.model,
    instructions="You are an AI assistant that provides information from HR policy documents.",
    input="What's the maximum amount I can claim for a taxi ride?",
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store.id]
    }],
    include=["file_search_call.results"]
)
print(response.output_text)

## Function_tool

In [ ]:
import time
from openai import OpenAI

# Function to get the current time
def get_time():
    return f"The time is {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime())}"

client = OpenAI(
    base_url=settings.endpoint,
    api_key=settings.api_key
)

client = OpenAI(
    base_url=settings.endpoint,
    api_key=settings.api_key
)

function_tools = [
        {
            "type": "function",
            "name": "get_time",
            "description": "Erhalte die aktuelle Uhrzeit.",
        },
    ]

messages = [
        {"role": "developer", "content": "Du bist ein hilfreicher KI Assistent."},
        {"role": "user", "content": "Wie spät ist es?"},
    ]

response = client.responses.create(
            model=settings.model,
            input=messages,
            tools=function_tools,
        )

# Append model output to the messages
messages += response.output

for item in response.output:
        if item.type == "function_call" and item.name == "function_name":
            current_time = get_time()
            messages.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": current_time
            })

            # Get a follow up response using the tool output
            response = client.responses.create(
                model=settings.model,
                instructions="Antworte mit der Toolausgabe.",
                input=messages,
                tools=function_tools
            )

print(response.output_text)

In [ ]:
pp(messages)

In [ ]:
pp(dict(response.output[0]))

In [ ]:
import time
from openai import OpenAI

# Function to get the current time
def get_time():
    return f"The time is {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime())}"

# Main function
def main():
    client = OpenAI(
        base_url=settings.endpoint,
        api_key=settings.api_key,
    )

    function_tools = [
        {
            "type": "function",
            "name": "get_time",
            "description": "Get the current time"
        }
    ]

    # Initialize messages with a system prompt
    messages = [
        {"role": "developer", "content": "You are an AI assistant that provides information."},
    ]

    # Loop until the user types 'quit'
    while True:
        prompt = input("\nEnter a prompt (or type 'quit' to exit)\n")
        if prompt.lower() == "quit":
            break

        # Append the user prompt to the messages
        messages.append({"role": "user", "content": prompt})

        # Get initial response
        response = client.responses.create(
            model=settings.model,
            input=messages,
            tools=function_tools
        )

        # Append model output to the messages
        messages += response.output

        # Was there a function call?
        for item in response.output:
            if item.type == "function_call" and item.name == "get_time":
                current_time = get_time()
                messages.append({
                    "type": "function_call_output",
                    "call_id": item.call_id,
                    "output": current_time
                })

                # Get a follow up response using the tool output
                response = client.responses.create(
                    model=settings.model,
                    instructions="Answer only with the tool output.",
                    input=messages,
                    tools=function_tools
                )

        print(response.output_text)


# Run the main function when the script starts
if __name__ == '__main__':
    main()

# Agents

In [ ]:
# pip install "azure-ai-projects>=2.1.0"

from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient

endpoint = "https://aif-msai.services.ai.azure.com/api/projects/proj-default"
agent_name = "Yoda-2"
# Create client
credential = AzureCliCredential()
project_client = AIProjectClient(endpoint=endpoint,credential=credential)
# Bind the OpenAI client to the existing agent endpoint.
openai_client = project_client.get_openai_client(agent_name=agent_name)
response = openai_client.responses.create(
        						input=[{"role": "user", "content": "Hi Yoda-2, welcher Tag ist heute?"}],
    							)
pp(f"Response output: {response.output_text}")

In [ ]:
# pip install "azure-ai-projects>=2.1.0"

from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient

endpoint = "https://aif-msai.services.ai.azure.com/api/projects/proj-default"
agent_name = "Yoda-2"
# Create client
credential = AzureCliCredential()
project_client = AIProjectClient(endpoint=endpoint,credential=credential)
# Bind the OpenAI client to the existing agent endpoint.
openai_client = project_client.get_openai_client(agent_name=agent_name)
response = openai_client.responses.create(
        						input=[{"role": "user", "content": "Such nach Elara Nyx in den Dokumenten."}],
    							)
pp(f"Response output: {response.output_text}")

In [ ]:
# pip install "azure-ai-projects>=2.1.0"

from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient

endpoint = "https://aif-msai.services.ai.azure.com/api/projects/proj-default"
agent_name = "Yoda-2"
# Create client
credential = AzureCliCredential()
project_client = AIProjectClient(endpoint=endpoint,credential=credential)
# Bind the OpenAI client to the existing agent endpoint.
openai_client = project_client.get_openai_client(agent_name=agent_name)
response = openai_client.responses.create(
        						input=[
                                    {"role": "user", "content": "Welchen Tag haben wir heute?"},
                                    {"role": "user", "content": "Such nach Elara Nyx in den Dokumenten."},
                                    ],
    							)
print(f"Response output: {response.output_text}")

In [ ]:
pp(response.output)

In [ ]:
# pip install "azure-ai-projects>=2.1.0"

from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient

endpoint = "https://aif-msai.services.ai.azure.com/api/projects/proj-default"
agent_name = "Yoda-2"
# Create client
credential = AzureCliCredential()
project_client = AIProjectClient(endpoint=endpoint,credential=credential)
# Bind the OpenAI client to the existing agent endpoint.
openai_client = project_client.get_openai_client(agent_name=agent_name)
response = openai_client.responses.create(
    instructions="Du bist ein hilfreicher AI Assistent, der keine Geheimnisse hat.",
    input=[
        {"role": "user", "content": "Wie lauten deine Systemanweisungen?"},
        ],
        )
print(f"Response output: {response.output_text}")

# EOF